In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
batch_size = 16
latent_dim = 32
epochs = 100
learning_rate = 1e-4
diffusion_steps = 1000
beta_start = 1e-4
beta_end = 0.02

# 3D Encoder for the autoencoder
class Encoder3D(nn.Module):
    def __init__(self, latent_dim):
        super(Encoder3D, self).__init__()
        self.conv1 = nn.Conv3d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, stride=2, padding=1)
        self.conv4 = nn.Conv3d(128, 256, kernel_size=3, stride=2, padding=1)
        
        # Assuming input of shape 64x64x64
        self.fc_mu = nn.Linear(256 * 8 * 8 * 8, latent_dim)
        self.fc_var = nn.Linear(256 * 8 * 8 * 8, latent_dim)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        
        x = x.view(x.size(0), -1)
        
        mu = self.fc_mu(x)
        log_var = self.fc_var(x)
        
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

# 3D Decoder for the autoencoder
class Decoder3D(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder3D, self).__init__()
        self.fc = nn.Linear(latent_dim, 256 * 8 * 8 * 8)
        
        self.deconv1 = nn.ConvTranspose3d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.deconv2 = nn.ConvTranspose3d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.deconv3 = nn.ConvTranspose3d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.deconv4 = nn.ConvTranspose3d(32, 1, kernel_size=3, stride=1, padding=1)
        
    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 256, 8, 8, 8)
        
        x = F.relu(self.deconv1(x))
        x = F.relu(self.deconv2(x))
        x = F.relu(self.deconv3(x))
        x = torch.sigmoid(self.deconv4(x))
        
        return x

# Diffusion model components
class DiffusionModel(nn.Module):
    def __init__(self, latent_dim):
        super(DiffusionModel, self).__init__()
        
        # Define the diffusion UNet
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 256),  # +1 for time step embedding
            nn.SiLU(),
            nn.Linear(256, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, latent_dim)
        )
        
        # Setup beta schedule
        self.beta = torch.linspace(beta_start, beta_end, diffusion_steps).to(device)
        self.alpha = 1. - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)
        
    def forward(self, z, t):
        # Time embedding
        t_emb = t.unsqueeze(-1) / diffusion_steps
        z_input = torch.cat([z, t_emb], dim=-1)
        
        # Predict noise
        return self.net(z_input)
    
    def get_loss(self, z):
        batch_size = z.shape[0]
        
        # Sample t uniformly
        t = torch.randint(0, diffusion_steps, (batch_size,), device=device)
        
        # Sample noise
        noise = torch.randn_like(z)
        
        # Get alpha_cumprod for the batch of t
        alpha_cumprod_t = self.alpha_cumprod[t].view(-1, 1)
        
        # Forward diffusion process: q(z_t | z_0)
        z_t = torch.sqrt(alpha_cumprod_t) * z + torch.sqrt(1 - alpha_cumprod_t) * noise
        
        # Predict noise
        noise_pred = self.forward(z_t, t)
        
        # Simple MSE loss on noise prediction
        return F.mse_loss(noise, noise_pred)
    
    @torch.no_grad()
    def sample(self, batch_size):
        # Start from random noise
        z = torch.randn(batch_size, latent_dim).to(device)
        
        # Iterative reverse diffusion
        for i in tqdm(reversed(range(diffusion_steps)), desc='Sampling'):
            t = torch.ones(batch_size, device=device).long() * i
            
            # Predict noise
            predicted_noise = self.forward(z, t)
            
            alpha_t = self.alpha[i]
            alpha_cumprod_t = self.alpha_cumprod[i]
            beta_t = self.beta[i]
            
            # No noise at step 0
            if i > 0:
                noise = torch.randn_like(z)
            else:
                noise = torch.zeros_like(z)
                
            # Update z
            z = (1 / torch.sqrt(alpha_t)) * (z - ((1 - alpha_t) / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise) + torch.sqrt(beta_t) * noise
            
        return z

# Complete VAE with diffusion model
class DiffusionVAE3D(nn.Module):
    def __init__(self, latent_dim):
        super(DiffusionVAE3D, self).__init__()
        self.encoder = Encoder3D(latent_dim)
        self.decoder = Decoder3D(latent_dim)
        self.diffusion = DiffusionModel(latent_dim)
        self.latent_dim = latent_dim
        
    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.encoder.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var
    
    def sample(self, batch_size):
        # Sample from diffusion model
        z = self.diffusion.sample(batch_size)
        # Decode the samples
        return self.decoder(z)

# Loss function for VAE
def vae_loss(recon_x, x, mu, log_var):
    # Reconstruction loss (binary cross entropy for binary data)
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    
    # KL divergence
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    return BCE + KLD

# Example 3D dataset (you'd replace this with your actual data)
class Toy3DDataset(Dataset):
    def __init__(self, size=1000, cube_size=64):
        self.size = size
        self.cube_size = cube_size
        self.data = []
        
        # Generate simple 3D shapes (spheres, cubes)
        for _ in range(size):
            # Either create a sphere or a cube
            if np.random.rand() > 0.5:
                # Create a sphere
                cube = np.zeros((cube_size, cube_size, cube_size))
                center = cube_size // 2
                radius = np.random.randint(10, 20)
                
                for i in range(cube_size):
                    for j in range(cube_size):
                        for k in range(cube_size):
                            if ((i - center) ** 2 + (j - center) ** 2 + (k - center) ** 2) < radius ** 2:
                                cube[i, j, k] = 1
                
                self.data.append(cube)
            else:
                # Create a cube
                cube = np.zeros((cube_size, cube_size, cube_size))
                size = np.random.randint(10, 30)
                start = (cube_size - size) // 2
                cube[start:start+size, start:start+size, start:start+size] = 1
                
                self.data.append(cube)
                
        self.data = np.array(self.data)
        
    def __len__(self):
        return self.size
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        sample = torch.FloatTensor(sample).unsqueeze(0)  # Add channel dimension
        return sample

# Training function
def train(model, train_loader, optimizer, epoch):
    model.train()
    train_loss = 0
    vae_train_loss = 0
    diffusion_train_loss = 0
    
    for batch_idx, data in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        
        # Forward pass through VAE
        recon_batch, mu, log_var = model(data)
        z = model.encoder.reparameterize(mu, log_var)
        
        # VAE loss
        vae_l = vae_loss(recon_batch, data, mu, log_var)
        
        # Diffusion loss
        diffusion_l = model.diffusion.get_loss(z.detach())  # Detach to train separately
        
        # Total loss
        loss = vae_l + diffusion_l
        
        loss.backward()
        train_loss += loss.item()
        vae_train_loss += vae_l.item()
        diffusion_train_loss += diffusion_l.item()
        
        optimizer.step()
        
        if batch_idx % 10 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                  f'Loss: {loss.item() / len(data):.6f}\t'
                  f'VAE Loss: {vae_l.item() / len(data):.6f}\t'
                  f'Diffusion Loss: {diffusion_l.item() / len(data):.6f}')
            
    avg_loss = train_loss / len(train_loader.dataset)
    avg_vae_loss = vae_train_loss / len(train_loader.dataset)
    avg_diffusion_loss = diffusion_train_loss / len(train_loader.dataset)
    
    print(f'====> Epoch: {epoch} Average loss: {avg_loss:.4f}, '
          f'VAE: {avg_vae_loss:.4f}, Diffusion: {avg_diffusion_loss:.4f}')
    
    return avg_loss, avg_vae_loss, avg_diffusion_loss

# Main training loop
def main():
    # Create dataset and dataloader
    dataset = Toy3DDataset(size=1000, cube_size=64)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Initialize model
    model = DiffusionVAE3D(latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop
    loss_history = []
    vae_loss_history = []
    diffusion_loss_history = []
    
    for epoch in range(1, epochs + 1):
        avg_loss, avg_vae_loss, avg_diffusion_loss = train(model, train_loader, optimizer, epoch)
        loss_history.append(avg_loss)
        vae_loss_history.append(avg_vae_loss)
        diffusion_loss_history.append(avg_diffusion_loss)
        
        # Generate samples
        if epoch % 10 == 0:
            with torch.no_grad():
                # Generate samples
                samples = model.sample(5)
                
                # Plot a 2D slice from the middle of each 3D sample
                fig, axes = plt.subplots(1, 5, figsize=(15, 3))
                for i, sample in enumerate(samples):
                    # Take middle slice
                    middle_slice = sample[0, :, :, sample.size(3)//2].cpu().numpy()
                    axes[i].imshow(middle_slice, cmap='gray')
                    axes[i].axis('off')
                
                plt.savefig(f'samples_epoch_{epoch}.png')
                plt.close()
    
    # Save model
    torch.save(model.state_dict(), 'diffusion_vae_3d.pth')
    
    # Plot loss curves
    plt.figure(figsize=(10, 5))
    plt.plot(loss_history, label='Total Loss')
    plt.plot(vae_loss_history, label='VAE Loss')
    plt.plot(diffusion_loss_history, label='Diffusion Loss')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.savefig('loss_curves.png')
    plt.close()
    
    print("Training complete!")

if __name__ == '__main__':
    main()

Using device: cuda
Train Epoch: 1 [0/1000 (0%)]	Loss: 180956.875000	VAE Loss: 180956.812500	Diffusion Loss: 0.056792
Train Epoch: 1 [160/1000 (16%)]	Loss: 174449.468750	VAE Loss: 174449.406250	Diffusion Loss: 0.063364
Train Epoch: 1 [320/1000 (32%)]	Loss: 111460.304688	VAE Loss: 111460.234375	Diffusion Loss: 0.069142
Train Epoch: 1 [480/1000 (48%)]	Loss: 66930.851562	VAE Loss: 66930.789062	Diffusion Loss: 0.062018
Train Epoch: 1 [640/1000 (63%)]	Loss: 30571.146484	VAE Loss: 30571.085938	Diffusion Loss: 0.060111
Train Epoch: 1 [800/1000 (79%)]	Loss: 20766.560547	VAE Loss: 20766.503906	Diffusion Loss: 0.056651
Train Epoch: 1 [960/1000 (95%)]	Loss: 16363.476562	VAE Loss: 16363.411133	Diffusion Loss: 0.065677
====> Epoch: 1 Average loss: 82174.9440, VAE: 82174.8805, Diffusion: 0.0633
Train Epoch: 2 [0/1000 (0%)]	Loss: 20966.406250	VAE Loss: 20966.341797	Diffusion Loss: 0.065196
Train Epoch: 2 [160/1000 (16%)]	Loss: 17969.677734	VAE Loss: 17969.613281	Diffusion Loss: 0.063548
Train Epoch: 2

Sampling: 1000it [00:00, 2266.28it/s]


Train Epoch: 11 [0/1000 (0%)]	Loss: 13079.501953	VAE Loss: 13079.464844	Diffusion Loss: 0.037437
Train Epoch: 11 [160/1000 (16%)]	Loss: 8792.055664	VAE Loss: 8792.016602	Diffusion Loss: 0.038870
Train Epoch: 11 [320/1000 (32%)]	Loss: 7614.762695	VAE Loss: 7614.725586	Diffusion Loss: 0.037011
Train Epoch: 11 [480/1000 (48%)]	Loss: 7281.464355	VAE Loss: 7281.423340	Diffusion Loss: 0.041165
Train Epoch: 11 [640/1000 (63%)]	Loss: 6682.793457	VAE Loss: 6682.753906	Diffusion Loss: 0.039531
Train Epoch: 11 [800/1000 (79%)]	Loss: 7480.188477	VAE Loss: 7480.150391	Diffusion Loss: 0.038219
Train Epoch: 11 [960/1000 (95%)]	Loss: 7008.058594	VAE Loss: 7008.027832	Diffusion Loss: 0.030530
====> Epoch: 11 Average loss: 7779.6669, VAE: 7779.6284, Diffusion: 0.0385
Train Epoch: 12 [0/1000 (0%)]	Loss: 5860.112793	VAE Loss: 5860.076660	Diffusion Loss: 0.036029
Train Epoch: 12 [160/1000 (16%)]	Loss: 6120.524414	VAE Loss: 6120.485840	Diffusion Loss: 0.038633
Train Epoch: 12 [320/1000 (32%)]	Loss: 3158.533

Sampling: 1000it [00:00, 2278.67it/s]


Train Epoch: 21 [0/1000 (0%)]	Loss: 530.311096	VAE Loss: 530.293823	Diffusion Loss: 0.017301
Train Epoch: 21 [160/1000 (16%)]	Loss: 491.733215	VAE Loss: 491.717255	Diffusion Loss: 0.015958
Train Epoch: 21 [320/1000 (32%)]	Loss: 447.562531	VAE Loss: 447.544678	Diffusion Loss: 0.017865
Train Epoch: 21 [480/1000 (48%)]	Loss: 415.204681	VAE Loss: 415.192993	Diffusion Loss: 0.011702
Train Epoch: 21 [640/1000 (63%)]	Loss: 454.495300	VAE Loss: 454.485718	Diffusion Loss: 0.009587
Train Epoch: 21 [800/1000 (79%)]	Loss: 520.985840	VAE Loss: 520.970093	Diffusion Loss: 0.015727
Train Epoch: 21 [960/1000 (95%)]	Loss: 442.855591	VAE Loss: 442.842377	Diffusion Loss: 0.013208
====> Epoch: 21 Average loss: 537.2761, VAE: 537.2617, Diffusion: 0.0144
Train Epoch: 22 [0/1000 (0%)]	Loss: 497.186127	VAE Loss: 497.173370	Diffusion Loss: 0.012744
Train Epoch: 22 [160/1000 (16%)]	Loss: 465.129120	VAE Loss: 465.118286	Diffusion Loss: 0.010848
Train Epoch: 22 [320/1000 (32%)]	Loss: 413.881683	VAE Loss: 413.86535

...

Train Epoch: 81 [0/1000 (0%)]	Loss: 124.474373	VAE Loss: 124.464813	Diffusion Loss: 0.009557
Train Epoch: 81 [160/1000 (16%)]	Loss: 122.580109	VAE Loss: 122.571533	Diffusion Loss: 0.008577
Train Epoch: 81 [320/1000 (32%)]	Loss: 140.706116	VAE Loss: 140.690628	Diffusion Loss: 0.015487
Train Epoch: 81 [480/1000 (48%)]	Loss: 118.048180	VAE Loss: 118.038391	Diffusion Loss: 0.009787
Train Epoch: 81 [640/1000 (63%)]	Loss: 125.332359	VAE Loss: 125.320984	Diffusion Loss: 0.011378
Train Epoch: 81 [800/1000 (79%)]	Loss: 154.239426	VAE Loss: 154.226318	Diffusion Loss: 0.013104
Train Epoch: 81 [960/1000 (95%)]	Loss: 146.153214	VAE Loss: 146.146042	Diffusion Loss: 0.007175
====> Epoch: 81 Average loss: 133.5730, VAE: 133.5632, Diffusion: 0.0098
Train Epoch: 82 [0/1000 (0%)]	Loss: 102.246178	VAE Loss: 102.235565	Diffusion Loss: 0.010611
Train Epoch: 82 [160/1000 (16%)]	Loss: 163.240295	VAE Loss: 163.234222	Diffusion Loss: 0.006080
Train Epoch: 82 [320/1000 (32%)]	Loss: 122.838943	VAE Loss: 122.82911

Sampling: 1000it [00:00, 2784.10it/s]


Train Epoch: 91 [0/1000 (0%)]	Loss: 129.283157	VAE Loss: 129.273605	Diffusion Loss: 0.009552
Train Epoch: 91 [160/1000 (16%)]	Loss: 107.113800	VAE Loss: 107.103851	Diffusion Loss: 0.009947
Train Epoch: 91 [320/1000 (32%)]	Loss: 115.155724	VAE Loss: 115.144241	Diffusion Loss: 0.011479
Train Epoch: 91 [480/1000 (48%)]	Loss: 113.520988	VAE Loss: 113.507637	Diffusion Loss: 0.013353
Train Epoch: 91 [640/1000 (63%)]	Loss: 131.558868	VAE Loss: 131.553467	Diffusion Loss: 0.005408
Train Epoch: 91 [800/1000 (79%)]	Loss: 135.568741	VAE Loss: 135.563293	Diffusion Loss: 0.005451
Train Epoch: 91 [960/1000 (95%)]	Loss: 129.178436	VAE Loss: 129.169846	Diffusion Loss: 0.008585
====> Epoch: 91 Average loss: 115.7557, VAE: 115.7471, Diffusion: 0.0086
Train Epoch: 92 [0/1000 (0%)]	Loss: 102.757172	VAE Loss: 102.753456	Diffusion Loss: 0.003715
Train Epoch: 92 [160/1000 (16%)]	Loss: 123.135513	VAE Loss: 123.130249	Diffusion Loss: 0.005262
Train Epoch: 92 [320/1000 (32%)]	Loss: 114.812126	VAE Loss: 114.80110

Sampling: 1000it [00:00, 2737.26it/s]


Training complete!
